# Part I — Foundations
### Tokenization, Embeddings, Neural Nets, Autoencoders

Welcome. This is the first notebook of an interactive curriculum on how LLMs actually work. No hand-waving, no "it's just a big neural net, don't worry about it." We're going to build everything from scratch, starting from the dumbest possible question and working our way up.

The ground rule: every idea gets code. If we write an equation, we implement it immediately, usually in a cell right below it. You should be running cells, breaking them, changing numbers, and watching what happens. Reading this notebook passively is a waste of your time — you have the LLM's output for that.

By the end of this notebook you will have:

- Implemented Byte-Pair Encoding from scratch (the same algorithm GPT-4 uses)
- Trained word embeddings and seen the famous `king - man + woman ≈ queen` trick work with your own eyes
- Built a 2-layer neural network in pure NumPy, derived its backprop by hand, and watched it learn XOR
- Trained an autoencoder on MNIST and strolled through its latent space while digits morphed between each other

About 45–60 minutes if you engage with it. Longer if you go down rabbit holes, which you should.

Let's go.

## Setup

Run this first. If a package is missing, uncomment the pip line. Everything here is cheap — no GPU required, no model downloads larger than a few MB.

In [ ]:
# !pip install -q numpy matplotlib tiktoken scikit-learn torch torchvision

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, ListedColormap

# A small aesthetic choice: default matplotlib blue is overused. Let's pick a palette.
PALETTE = {
    "ink":     "#1a1a2e",
    "paper":   "#f7f3e9",
    "rose":    "#e63946",
    "amber":   "#f4a261",
    "teal":    "#2a9d8f",
    "indigo":  "#3d5a80",
    "plum":    "#7b2cbf",
    "lime":    "#a8dadc",
}

plt.rcParams.update({
    "figure.facecolor": PALETTE["paper"],
    "axes.facecolor":   PALETTE["paper"],
    "axes.edgecolor":   PALETTE["ink"],
    "axes.labelcolor":  PALETTE["ink"],
    "xtick.color":      PALETTE["ink"],
    "ytick.color":      PALETTE["ink"],
    "text.color":       PALETTE["ink"],
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

rng = np.random.default_rng(1337)
print("setup ok")

---

# 1. Text → Numbers

## Why can't computers read words?

Here's the thing. A computer is, deep down, a box that multiplies numbers together really fast. That's all it can do. When you type "hello" into ChatGPT, somewhere between your keyboard and the GPU, those five letters have to become numbers. Not metaphorically — literally. A list of integers that the model can look up in a table and start doing linear algebra with.

So the first problem in building a language model isn't attention or transformers or any of the sexy stuff. It's much more boring: **how do we turn text into numbers?**

The obvious answer is "give every word a unique ID." Dictionary-style. `cat → 42`, `dog → 43`, and so on. This is called word-level tokenization and it sucks for two reasons:

1. The vocabulary is enormous. English has hundreds of thousands of words, and that's before you count typos, proper nouns, and Twitter.
2. It can't handle new words. The moment someone writes "skibidi" your model shrugs.

The other extreme is character-level: `h=0, e=1, l=2, l=2, o=3`. Tiny vocabulary (~256 bytes), handles anything you throw at it, but sequences become brutally long and the model has to re-learn that "the" is one concept from scratch every time.

What everyone uses in practice is a beautiful middle ground called **Byte-Pair Encoding**. The algorithm is so dumb it's insulting, and yet it is what powers GPT, Claude, Llama, all of them. Let's build it.

## BPE, the world's laziest compression algorithm

BPE was originally a data compression trick from 1994. Philip Gage published it in a C programming magazine. Nobody cared for twenty years. Then someone tried it on text for neural translation and now it runs the world.

The algorithm, in English:

> Start with characters. Find the pair of adjacent symbols that occurs most often. Merge it into a new single symbol. Repeat until you have as many symbols as you want.

That's literally it. There's no machine learning. It's a counting loop.

Let's implement it.

In [ ]:
from collections import Counter

# Our "corpus" — a tiny training set. In real life this is trillions of tokens.
corpus = (
    "the cat sat on the mat "
    "the cat sat on the hat "
    "the bat sat on the rat "
    "the rat ate the cheese "
    "the cat ate the rat "
    "the dog chased the cat "
    "the cat chased the mouse "
    "the mouse hid from the cat"
)

# Step 0: split into "words" and represent each as a tuple of characters,
# with a special end-of-word marker so the tokenizer can tell word boundaries.
def word_to_symbols(word):
    return tuple(list(word) + ["</w>"])

words = corpus.split()
vocab = Counter(words)
# vocab maps word -> frequency; we'll work on the symbol representation
symbol_vocab = {word_to_symbols(w): c for w, c in vocab.items()}

for k, v in list(symbol_vocab.items())[:5]:
    print(v, k)

Right. Each word is a tuple of single characters plus `</w>`. Now the core move: find the most frequent adjacent pair across the whole corpus.

In [ ]:
def get_pair_counts(symbol_vocab):
    pairs = Counter()
    for symbols, freq in symbol_vocab.items():
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs

pair_counts = get_pair_counts(symbol_vocab)
print("top 5 pairs:")
for pair, c in pair_counts.most_common(5):
    print(f"  {pair}: {c}")

The top pair is whatever appears next to itself the most. In our corpus, `t` followed by `h` (as in "the") is going to dominate. Now we *merge* that pair — wherever we see `t` immediately followed by `h`, we replace them with a single new symbol `th`.

In [ ]:
def merge_pair(symbol_vocab, pair):
    a, b = pair
    merged = a + b
    new_vocab = {}
    for symbols, freq in symbol_vocab.items():
        new_symbols = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i+1] == b:
                new_symbols.append(merged)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        new_vocab[tuple(new_symbols)] = freq
    return new_vocab

# Do ONE merge, see what happens
top_pair = pair_counts.most_common(1)[0][0]
print("merging:", top_pair)
symbol_vocab_after = merge_pair(symbol_vocab, top_pair)
for k, v in list(symbol_vocab_after.items())[:5]:
    print(v, k)

Notice how `('t', 'h', 'e', '</w>')` became `('th', 'e', '</w>')`. We've created a new symbol. That's the entire idea.

Now we just do it in a loop. Each iteration merges the current most frequent pair and learns one new BPE "merge rule". After N merges we have a tokenizer.

In [ ]:
def train_bpe(corpus, num_merges, verbose=False):
    words = corpus.split()
    vocab = Counter(words)
    symbol_vocab = {word_to_symbols(w): c for w, c in vocab.items()}

    merges = []  # ordered list of merge rules
    for step in range(num_merges):
        pairs = get_pair_counts(symbol_vocab)
        if not pairs:
            break
        best = pairs.most_common(1)[0][0]
        symbol_vocab = merge_pair(symbol_vocab, best)
        merges.append(best)
        if verbose:
            print(f"step {step:3d}: merge {best} -> {best[0]+best[1]}")
    return merges, symbol_vocab

merges, final_vocab = train_bpe(corpus, num_merges=20, verbose=True)

Watch the output. The first merges pick up frequent things like `t+h -> th`, then `th + e -> the`. The BPE algorithm is literally discovering that "the" is a thing, by counting.

Now the encoder. Given a new word, how do we tokenize it? We apply the merge rules *in the order they were learned*. Earlier merges (more frequent patterns) take priority.

In [ ]:
def encode(word, merges):
    symbols = list(word) + ["</w>"]
    for a, b in merges:
        i = 0
        new_symbols = []
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i+1] == b:
                new_symbols.append(a + b)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols
    return symbols

for word in ["the", "cat", "cheese", "skibidi"]:
    print(f"{word:10s} -> {encode(word, merges)}")

Look at "skibidi" — a word the tokenizer never saw. It falls back to individual characters. That's the graceful degradation that makes BPE magical: any string in the world can be tokenized, because the absolute worst case is single bytes.

Meanwhile "the" collapsed to a single token `the</w>` because it was everywhere in training. Common words get short representations, rare words get long ones. It's an information-theoretic win.

## Now the real thing: tiktoken

That's what's happening under the hood. Let's use the production version — `tiktoken`, OpenAI's actual BPE implementation. Same algorithm, but trained on hundreds of gigabytes of internet text with a vocabulary of ~100k tokens.

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")  # the GPT-4 encoding

sentence = "The quick brown fox jumps over the lazy dog."
ids = enc.encode(sentence)
print("ids:", ids)
print("decoded:", [enc.decode([i]) for i in ids])
print(f"{len(ids)} tokens for {len(sentence)} chars")

## The surprise: "unhappiness" is three tokens, "cat" is one

This is one of those things that feels obvious once you see it but is weird the first time. Let's look at what the tokenizer does to morphologically complex words.

In [ ]:
test_words = [
    "cat", "cats", "unhappiness", "antidisestablishmentarianism",
    "GPT", "GPT-4", "supercalifragilisticexpialidocious",
    "hello", "hellohello", "🚀", "你好",
]

for w in test_words:
    ids = enc.encode(w)
    pieces = [enc.decode([i]) for i in ids]
    print(f"{w:40s} -> {len(ids)} tokens  {pieces}")

Couple of things to notice.

`cat` is one token because it's one of the most common English words ever. `cats` is *also* one token — BPE learned the plural as a unit. But `unhappiness` is three: `un`, `happ`, `iness`. BPE discovered that `un` is a frequent prefix and `iness` is a frequent suffix, but "unhappiness" as a whole wasn't common enough to earn its own merge. The model sees the word compositionally, through its morphemes, which is actually pretty elegant.

Chinese characters usually end up as 2–3 tokens each because the tokenizer was trained mostly on English — each Chinese character is multiple UTF-8 bytes, and each byte is often its own fallback token. That's why Chinese API calls cost more per character.

Emoji are similarly fragmented. 🚀 is a multi-byte UTF-8 sequence that BPE had to break up.

## Visualizing tokens

Let's make the tokenization literally visible. We'll color each token a different shade and print the sentence with token boundaries.

In [ ]:
from IPython.display import HTML, display

def render_tokens(text, encoding_name="cl100k_base"):
    enc = tiktoken.get_encoding(encoding_name)
    ids = enc.encode(text)
    colors = ["#e63946", "#f4a261", "#2a9d8f", "#3d5a80", "#7b2cbf", "#a8dadc"]
    html = ""
    for idx, tok_id in enumerate(ids):
        piece = enc.decode([tok_id]).replace(" ", "&nbsp;").replace("\n", "\\n")
        color = colors[idx % len(colors)]
        html += (
            f'<span style="background:{color};color:#1a1a2e;padding:2px 4px;'
            f'margin:1px;border-radius:3px;font-family:monospace;">{piece}</span>'
        )
    html += f'<div style="margin-top:8px;color:#666;font-family:monospace;">{len(ids)} tokens</div>'
    return HTML(html)

display(render_tokens("The quick brown fox jumps over the lazy dog."))
display(render_tokens("Tokenization is surprisingly nontrivial."))
display(render_tokens("    def hello_world():\n        return 42"))

Notice what happens with the code example. Leading whitespace gets its own tokens, and the tokenizer has special tokens for common indentation patterns like `    ` (four spaces). This is why LLMs write Python well — their tokenizer was explicitly tuned for it.

## Vocabulary size is a trade-off

More tokens in your vocabulary = shorter sequences (good, cheaper to process) but larger embedding tables (bad, more parameters). Let's make this concrete.

In [ ]:
# Compare several encodings: the same text, different vocab sizes
import tiktoken

sample_text = (
    "The transformer architecture revolutionized natural language processing "
    "by replacing recurrence with self-attention, enabling massive parallelism "
    "and better long-range dependency modeling." * 20
)

encodings = [
    ("r50k_base",   "GPT-2 (50k)"),
    ("p50k_base",   "GPT-3 (50k)"),
    ("cl100k_base", "GPT-4 (100k)"),
    ("o200k_base",  "GPT-4o (200k)"),
]

results = []
for name, label in encodings:
    try:
        e = tiktoken.get_encoding(name)
        n = len(e.encode(sample_text))
        results.append((label, n, e.n_vocab))
    except Exception as ex:
        print(f"skipping {name}: {ex}")

fig, ax = plt.subplots(figsize=(9, 4.5))
labels = [r[0] for r in results]
ntoks  = [r[1] for r in results]
vocabs = [r[2] for r in results]

bars = ax.barh(labels, ntoks, color=[PALETTE["rose"], PALETTE["amber"], PALETTE["teal"], PALETTE["indigo"]])
for bar, nv, nt in zip(bars, vocabs, ntoks):
    ax.text(bar.get_width() + 3, bar.get_y() + bar.get_height()/2,
            f"{nt} tokens   (vocab={nv:,})", va="center", fontsize=10)
ax.set_xlabel("tokens used for the same paragraph")
ax.set_title("Bigger vocab → fewer tokens for the same text", loc="left", fontweight="bold")
ax.set_xlim(0, max(ntoks) * 1.45)
plt.tight_layout()
plt.show()

The GPT-4o tokenizer (200k vocab) needs ~30% fewer tokens than the old GPT-2 tokenizer for the same English text. Over a multi-billion-token pretraining run, that adds up to real money. It's also part of why modern models feel faster — less to process per message.

**Checkpoint:**

1. Tokenize your own name with `enc.encode`. How many tokens is it? Why?
2. Find a word where adding one letter *reduces* the token count. (Hint: try adding `s` to common nouns.)
3. Why does Chinese text cost more per character than English?

---

# 2. Embeddings — Meaning as Geometry

Token IDs are fine for storage but useless for learning. The integer `42` tells the model nothing about what `cat` means or which other tokens it's similar to. ID `42` and ID `43` could be "cat" and "dog", or "cat" and "photosynthesis" — the numbers carry no signal.

So we do something slightly crazy. We give every token a **vector**. A list of, say, 300 numbers. And we let the model *learn* what those numbers should be, such that tokens with similar meanings end up with similar vectors.

This is called an embedding, and it is the quiet protagonist of deep learning. Everything downstream — attention, transformers, GPT-4 — is operating on these vectors, not on words.

The wild claim is that **meaning becomes geometry**. Semantic similarity turns into spatial closeness. Analogies turn into vector arithmetic. Let's see it work.

## The lookup operation, demystified

An embedding layer is a matrix $E \in \mathbb{R}^{V \times d}$ where $V$ is your vocabulary size and $d$ is the embedding dimension. To embed token $i$, you literally grab row $i$:

$$
E(i) = E[i, :] \in \mathbb{R}^d
$$

Read that again. It's a lookup. There is no magic. The "learning" happens because we allow gradient descent to modify the rows of $E$ during training, and somehow the rows that end up similar are the ones whose tokens appeared in similar contexts. Which is a whole other thing we'll get to.

Let's just implement the lookup to drive home how stupidly simple it is.

In [ ]:
# A toy embedding matrix: 10 words, 4-dimensional embeddings
vocab = ["the", "cat", "dog", "bird", "fish", "runs", "swims", "flies", "sits", "eats"]
V, d = len(vocab), 4

# Random init — in a real model these get trained.
E = rng.normal(0, 0.5, size=(V, d))

word_to_id = {w: i for i, w in enumerate(vocab)}

def embed(word):
    return E[word_to_id[word]]

print("cat embedding:", embed("cat"))
print("dog embedding:", embed("dog"))
print("shape:", E.shape)

That's the embedding layer. A matrix and a lookup. If you ever read a paper that says "we project the tokens into a 4096-dimensional embedding space" — this is what they mean. They have a 4096-column matrix and they're indexing into it.

## Cosine similarity

Okay but how do we measure whether two embeddings are "close"? You might reach for Euclidean distance — but in high dimensions, raw distances get weird. Long vectors are far from everything regardless of direction. What actually matters for meaning is *direction*.

So we use cosine similarity:

$$
\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \, \|\mathbf{b}\|}
$$

Unpacking that:

- $\mathbf{a} \cdot \mathbf{b}$ is the dot product. Elementwise multiply, then sum. When two vectors point the same way, this is big. When they're perpendicular, it's zero. When they point opposite, it's negative.
- $\|\mathbf{a}\|$ is the length (Euclidean norm) of $\mathbf{a}$. Dividing by the lengths strips out magnitude and leaves only direction.
- The whole thing is in $[-1, 1]$. 1 means "same direction". 0 means "orthogonal, unrelated". -1 means "opposite".

In [ ]:
def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

# Sanity check: a vector is perfectly similar to itself
v = rng.normal(size=5)
print("cos(v, v) =", cosine(v, v))
print("cos(v, -v) =", cosine(v, -v))
print("cos(v, random) =", cosine(v, rng.normal(size=5)))

## Real embeddings: load GloVe-style vectors we trained ourselves

Rather than downloading a 1GB word2vec file, let's fabricate a mini GloVe. We'll hand-craft a tiny embedding space where semantic structure exists, so we can see the arithmetic work without the noise of a real 50k-word model. This is cheating pedagogically, but the geometry is real and every concept transfers.

(If you want the real thing, the code below shows how to swap in `gensim` pretrained vectors.)

In [ ]:
# Hand-crafted mini embeddings — we'll place words in a 2D plane so we can see
# the structure, then lift them to higher dimensions with noise.
# Axes (by construction): x = "royalty", y = "gender (male=+, female=-)"

_hand = {
    # royalty, gender
    "king":     ( 1.0,  1.0),
    "queen":    ( 1.0, -1.0),
    "man":      (-1.0,  1.0),
    "woman":    (-1.0, -1.0),
    "prince":   ( 0.7,  1.0),
    "princess": ( 0.7, -1.0),
    "boy":      (-1.0,  0.7),
    "girl":     (-1.0, -0.7),

    # country / capital — let's add a second plane
    # We'll use dims (royalty, gender, country_axis, city_axis)
}

# Extend to 4D. The royalty/gender words get zeros on the country axes.
def to4d(xy, extra=(0.0, 0.0)):
    return np.array([xy[0], xy[1], extra[0], extra[1]], dtype=np.float64)

words_4d = {w: to4d(xy) for w, xy in _hand.items()}

# Country/capital pairs — arrange them on the (country_axis, city_axis) plane
# with the *offset* from country to capital being the same vector.
country_capital = [
    ("france",  "paris"),
    ("germany", "berlin"),
    ("italy",   "rome"),
    ("japan",   "tokyo"),
    ("spain",   "madrid"),
]
offset = np.array([0.0, 0.0, 0.0, 1.2])   # "is_capital_of" direction
for i, (c, cap) in enumerate(country_capital):
    base = np.array([0.0, 0.0, -1.0 + 0.4*i, -0.6])
    words_4d[c]   = base
    words_4d[cap] = base + offset

# A tiny amount of noise so it's not pathologically clean
for k in words_4d:
    words_4d[k] = words_4d[k] + rng.normal(0, 0.05, size=4)

print(f"we have {len(words_4d)} words in a {next(iter(words_4d.values())).shape[0]}-dim space")

## The greatest party trick in NLP history

Mikolov's word2vec paper in 2013 showed that you could take word embeddings — just trained to predict neighboring words — and do *arithmetic* with them. The canonical example:

$$
\text{king} - \text{man} + \text{woman} \approx \text{queen}
$$

Read this carefully. We take the vector for "king", subtract "man" (roughly removing the male-ness), add "woman" (adding female-ness), and the resulting vector is closest to "queen". The geometry encoded the abstract concept of gender, and you can do algebra with it.

This was a shock to the field. Nobody told the model what gender was. It figured it out from how words co-occur in text. Let's reproduce it.

In [ ]:
def nearest(target_vec, vocab, top_k=5, exclude=()):
    scores = []
    for w, v in vocab.items():
        if w in exclude:
            continue
        scores.append((w, cosine(target_vec, v)))
    scores.sort(key=lambda x: -x[1])
    return scores[:top_k]

# The classic analogy
king  = words_4d["king"]
man   = words_4d["man"]
woman = words_4d["woman"]

target = king - man + woman
print("king - man + woman = ?")
for w, s in nearest(target, words_4d, top_k=5, exclude={"king", "man", "woman"}):
    print(f"  {w:10s}  cosine={s:+.3f}")

Boom. `queen` wins. With a real pretrained model (word2vec, GloVe, FastText) this actually works for thousands of analogies — `paris:france :: tokyo:?` recovers `japan`, `walking:walked :: swimming:?` recovers `swam`, etc. Try one with our country set:

In [ ]:
# paris - france + germany ≈ ?
target = words_4d["paris"] - words_4d["france"] + words_4d["germany"]
print("paris - france + germany = ?")
for w, s in nearest(target, words_4d, top_k=5, exclude={"paris", "france", "germany"}):
    print(f"  {w:10s}  cosine={s:+.3f}")

## Visualizing the geometry with PCA

Our embeddings live in 4D, which is hard to look at. PCA (Principal Component Analysis) finds the directions in the data that have the most variance and projects everything onto those. It's a way to take a blurry 4D photograph and summarize the two dimensions where the most interesting stuff is happening.

In [ ]:
from sklearn.decomposition import PCA

words = list(words_4d.keys())
X = np.stack([words_4d[w] for w in words])
X2 = PCA(n_components=2).fit_transform(X)

fig, ax = plt.subplots(figsize=(9, 7))

# Draw parallel-structure arrows for the country-capital pairs
for c, cap in country_capital:
    ci, capi = words.index(c), words.index(cap)
    ax.annotate("",
                xy=X2[capi], xytext=X2[ci],
                arrowprops=dict(arrowstyle="->", color=PALETTE["teal"], lw=2, alpha=0.7))

# Draw the king/queen/man/woman rectangle
royal_pairs = [("king", "queen"), ("man", "woman"), ("prince", "princess"), ("boy", "girl")]
for a, b in royal_pairs:
    ai, bi = words.index(a), words.index(b)
    ax.plot([X2[ai,0], X2[bi,0]], [X2[ai,1], X2[bi,1]],
            "--", color=PALETTE["rose"], alpha=0.5, lw=1.5)

# Scatter + labels
colors = []
for w in words:
    if w in {"king","queen","prince","princess","man","woman","boy","girl"}:
        colors.append(PALETTE["rose"])
    else:
        colors.append(PALETTE["teal"])

ax.scatter(X2[:,0], X2[:,1], s=120, c=colors, edgecolor=PALETTE["ink"], linewidth=1.5, zorder=3)
for i, w in enumerate(words):
    ax.annotate(w, X2[i], xytext=(8, 4), textcoords="offset points", fontsize=11)

ax.set_title("Embeddings projected to 2D via PCA", loc="left", fontweight="bold")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

Look at the picture. The red dashed lines (gender pairs) are roughly parallel to each other — that's the "gender direction" in the embedding space, a single linear axis that the model learned. The teal arrows (country → capital) are also roughly parallel, pointing in a different direction — the "is the capital of" relation is also a linear offset.

This is the thing that blew people's minds in 2013. The relationships between words became **linear offsets** in the vector space. Which is how the arithmetic trick works: when you do `king - man + woman`, you're taking king's position, stepping along the gender axis toward female, and landing on queen.

It turns out linear structure like this is not unique to words. It shows up in vision embeddings, audio embeddings, and modern LLM activations. Understanding it here is a big deal.

## t-SNE, for when PCA isn't enough

PCA is linear — it can only preserve distances along straight axes. For high-dimensional embeddings with nonlinear structure, t-SNE often gives prettier clusters. It's nonlinear, stochastic, and a little unreliable, but visually striking. Quick demo on our mini vocab:

In [ ]:
from sklearn.manifold import TSNE

# t-SNE needs more samples than 13 points to look interesting, so let's scramble up
# a larger toy corpus by adding noisy duplicates.
big_words = []
big_X = []
for w, v in words_4d.items():
    for _ in range(8):
        big_words.append(w)
        big_X.append(v + rng.normal(0, 0.1, size=4))
big_X = np.array(big_X)

X_tsne = TSNE(n_components=2, perplexity=10, random_state=42, init="pca").fit_transform(big_X)

fig, ax = plt.subplots(figsize=(9, 7))
unique = list(set(big_words))
color_map = {w: plt.cm.tab20(i / len(unique)) for i, w in enumerate(unique)}
for i, w in enumerate(big_words):
    ax.scatter(X_tsne[i,0], X_tsne[i,1], color=color_map[w], s=60, alpha=0.7,
               edgecolor=PALETTE["ink"], linewidth=0.5)
# Label one representative per cluster
seen = set()
for i, w in enumerate(big_words):
    if w not in seen:
        ax.annotate(w, X_tsne[i], fontsize=10, fontweight="bold")
        seen.add(w)
ax.set_title("t-SNE of the same embeddings (nonlinear projection)", loc="left", fontweight="bold")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

**Checkpoint:**

1. Pick another analogy. What should `boy - man + woman` give you? Try it.
2. What does it mean for two embeddings to have cosine similarity of 0? Of -1?
3. If we trained embeddings from scratch, where does the "gender axis" come from? (Hint: it's not in the code anywhere.)

---

# 3. Neural Networks 101

Time to build a brain. A small one. In NumPy, with no autograd, no PyTorch, nothing between us and the math. If you've only ever used a framework, this is the part where you go "oh, *that's* all it is."

A neural network is an embarrassingly simple object:

1. Multiply input by a weight matrix.
2. Add a bias.
3. Squash with a nonlinearity.
4. Repeat.

At the end you have a prediction. Compare it to the answer, compute a loss, and nudge every weight slightly in the direction that would have made the loss smaller. Do this a million times. Congratulations, you have GPT-4.

I'm being flippant but also not wrong. Every neural net — from your toy XOR classifier to Gemini Ultra — is this same recipe scaled up. Let's bake the cookie.

## The forward pass, in math

A single layer of a neural network does this:

$$
\mathbf{h} = \sigma(\mathbf{W}\mathbf{x} + \mathbf{b})
$$

Let's walk through it symbol by symbol:

- $\mathbf{x}$ is the input vector. Could be pixels, embeddings, features — doesn't matter.
- $\mathbf{W}$ is the weight matrix. If $\mathbf{x}$ has 2 elements and we want $\mathbf{h}$ to have 4, then $\mathbf{W}$ is 4×2.
- $\mathbf{W}\mathbf{x}$ is a matrix-vector multiply. Output: a vector of 4 numbers.
- $\mathbf{b}$ is a bias vector, one number per output. We just add it. It lets each neuron have its own "threshold".
- $\sigma$ is the nonlinearity. Sigmoid, tanh, ReLU — we'll use ReLU because it's what everyone uses and the math is trivial. $\text{ReLU}(x) = \max(0, x)$.

Stack two of these and you have a two-layer net:

$$
\hat{\mathbf{y}} = \mathbf{W}_2 \, \text{ReLU}(\mathbf{W}_1 \mathbf{x} + \mathbf{b}_1) + \mathbf{b}_2
$$

That final output $\hat{\mathbf{y}}$ is the network's prediction. Notice there's no nonlinearity on the *output* — we'll let a softmax or a loss function handle that separately.

In [ ]:
def relu(x):
    return np.maximum(0, x)

def relu_grad(x):
    # Derivative of ReLU is 1 where x>0, else 0
    return (x > 0).astype(x.dtype)

# A 2-layer net: input dim 2, hidden dim 8, output dim 2
def init_net(d_in=2, d_hidden=8, d_out=2, seed=0):
    r = np.random.default_rng(seed)
    # He initialization for ReLU
    W1 = r.normal(0, np.sqrt(2.0 / d_in), size=(d_hidden, d_in))
    b1 = np.zeros(d_hidden)
    W2 = r.normal(0, np.sqrt(2.0 / d_hidden), size=(d_out, d_hidden))
    b2 = np.zeros(d_out)
    return {"W1": W1, "b1": b1, "W2": W2, "b2": b2}

def forward(params, X, use_relu=True):
    # X has shape (N, d_in). Returns logits shape (N, d_out), plus a cache for backprop.
    W1, b1, W2, b2 = params["W1"], params["b1"], params["W2"], params["b2"]
    z1 = X @ W1.T + b1                           # (N, d_hidden)
    a1 = relu(z1) if use_relu else z1            # nonlinearity (or not, for the ablation)
    z2 = a1 @ W2.T + b2                          # (N, d_out)
    cache = (X, z1, a1, z2, use_relu)
    return z2, cache

net = init_net()
X_demo = np.array([[0.3, -1.2], [1.0, 0.5]])
out, _ = forward(net, X_demo)
print("forward output shape:", out.shape)
print(out)

## The loss

For classification with $C$ classes, the standard loss is **cross-entropy** on softmax probabilities:

$$
p_i = \frac{e^{z_i}}{\sum_{j=1}^{C} e^{z_j}}, \qquad L = -\sum_{i} y_i \log p_i
$$

In words:

- $z_i$ is the raw logit for class $i$ coming out of the network.
- Softmax turns logits into probabilities: exponentiate them (makes everything positive), divide by the sum (makes them sum to 1).
- $y_i$ is 1 for the correct class and 0 for all others (a one-hot target).
- The loss is $-\log(p_{\text{correct}})$ — how surprised the model was by the right answer. If the model put 99% probability on the right class, $-\log(0.99) \approx 0.01$, tiny loss. If it put 1%, $-\log(0.01) \approx 4.6$, painful loss.

Softmax + cross-entropy has a famously elegant gradient: $\partial L / \partial z = p - y$. This is not a coincidence — softmax and cross-entropy were literally designed to cancel nicely. We'll use this identity in backprop.

In [ ]:
def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)  # numerical stability
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

def cross_entropy(logits, y):
    # logits: (N, C), y: (N,) integer labels
    p = softmax(logits)
    N = logits.shape[0]
    # pick out the probability assigned to the true class for each example
    correct = p[np.arange(N), y]
    return -np.log(correct + 1e-12).mean(), p

# Quick test
logits = np.array([[2.0, 1.0, 0.1], [0.5, 2.5, 0.0]])
y = np.array([0, 1])
loss, probs = cross_entropy(logits, y)
print("probs:\n", probs.round(3))
print("loss:", loss)

## Backprop, by hand, one time so we understand it forever

Here we go. This is the part everyone fears and it's actually just the chain rule from calculus, mechanically applied.

We want to know $\partial L / \partial W_1$, $\partial L / \partial b_1$, $\partial L / \partial W_2$, $\partial L / \partial b_2$ — how the loss changes as we wiggle each parameter. Then we'll take a tiny step in the direction that decreases $L$, and repeat.

The trick is to compute gradients **backward** through the network, reusing intermediate results. Start at the output, work back to the input.

Step 1 — gradient of loss with respect to logits:

$$
\frac{\partial L}{\partial z_2} = \mathbf{p} - \mathbf{y}_{\text{one-hot}}
$$

Step 2 — logits come from $z_2 = a_1 W_2^T + b_2$, so:

$$
\frac{\partial L}{\partial W_2} = \frac{\partial L}{\partial z_2}^T a_1, \qquad \frac{\partial L}{\partial b_2} = \frac{\partial L}{\partial z_2}
$$

Step 3 — push the gradient back through $W_2$:

$$
\frac{\partial L}{\partial a_1} = \frac{\partial L}{\partial z_2} W_2
$$

Step 4 — through the ReLU: $a_1 = \text{ReLU}(z_1)$, so multiply by the ReLU derivative (1 where $z_1 > 0$, else 0):

$$
\frac{\partial L}{\partial z_1} = \frac{\partial L}{\partial a_1} \odot \mathbb{1}[z_1 > 0]
$$

Step 5 — through the first linear layer, same shape as step 2:

$$
\frac{\partial L}{\partial W_1} = \frac{\partial L}{\partial z_1}^T \mathbf{x}, \qquad \frac{\partial L}{\partial b_1} = \frac{\partial L}{\partial z_1}
$$

That's it. Five steps. Let's code it.

In [ ]:
def backward(params, cache, probs, y):
    X, z1, a1, z2, use_relu = cache
    N, C = probs.shape
    W2 = params["W2"]

    # Step 1: dL/dz2 = p - y_onehot
    dz2 = probs.copy()
    dz2[np.arange(N), y] -= 1.0
    dz2 /= N  # because our loss was an average

    # Step 2: gradients for W2, b2
    dW2 = dz2.T @ a1         # (C, d_hidden)
    db2 = dz2.sum(axis=0)    # (C,)

    # Step 3: push into a1
    da1 = dz2 @ W2           # (N, d_hidden)

    # Step 4: through ReLU (or not)
    if use_relu:
        dz1 = da1 * relu_grad(z1)
    else:
        dz1 = da1

    # Step 5: gradients for W1, b1
    dW1 = dz1.T @ X          # (d_hidden, d_in)
    db1 = dz1.sum(axis=0)

    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}

print("backward defined — this is our autograd.")

## Training on XOR — the impossible problem for a linear model

XOR is the "hello world" of neural networks because it's the simplest possible problem that can't be solved with a straight line.

| x1 | x2 | XOR |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

If you plot those four points with XOR as the color, the two classes are on opposite diagonals. No straight line separates them. A linear model literally cannot solve XOR — it was a famous 1969 result by Minsky and Papert that killed neural net research for a decade.

What saves us is the nonlinearity. With even one hidden layer and a nonlinearity, the network can bend space enough to separate the diagonals. Let's watch it happen.

In [ ]:
# XOR dataset — add a bit of noise so it's a real classification problem
def make_xor(n=400, noise=0.15, seed=0):
    r = np.random.default_rng(seed)
    X = r.uniform(-1, 1, size=(n, 2))
    y = ((X[:,0] > 0) ^ (X[:,1] > 0)).astype(int)
    X = X + r.normal(0, noise, size=X.shape)
    return X, y

X_train, y_train = make_xor(400, noise=0.15)

fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(X_train[y_train==0, 0], X_train[y_train==0, 1], color=PALETTE["rose"],  label="class 0", edgecolor=PALETTE["ink"])
ax.scatter(X_train[y_train==1, 0], X_train[y_train==1, 1], color=PALETTE["teal"],  label="class 1", edgecolor=PALETTE["ink"])
ax.set_title("XOR — the two classes are on opposite diagonals", loc="left", fontweight="bold")
ax.legend(); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

In [ ]:
def train(X, y, steps=2000, lr=0.3, use_relu=True, hidden=16, seed=0, snapshots_at=None):
    params = init_net(d_in=2, d_hidden=hidden, d_out=2, seed=seed)
    history = []
    snapshots = {}
    snapshots_at = snapshots_at or []
    for step in range(steps):
        logits, cache = forward(params, X, use_relu=use_relu)
        loss, probs  = cross_entropy(logits, y)
        grads = backward(params, cache, probs, y)
        for k in params:
            params[k] -= lr * grads[k]
        history.append(loss)
        if step in snapshots_at:
            snapshots[step] = {k: v.copy() for k, v in params.items()}
    snapshots[steps-1] = {k: v.copy() for k, v in params.items()}
    return params, history, snapshots

snap_at = [0, 50, 200, 1000]
final_params, loss_hist, snapshots = train(X_train, y_train, steps=2000, lr=0.3,
                                           use_relu=True, hidden=16, snapshots_at=snap_at)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(loss_hist, color=PALETTE["plum"], lw=2)
ax.set_xlabel("step"); ax.set_ylabel("cross-entropy loss")
ax.set_title("Training loss on XOR", loc="left", fontweight="bold")
ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

print(f"final loss: {loss_hist[-1]:.4f}")

## Watching the decision boundary form

The loss curve is information-dense but not visceral. What's actually happening is that the network is *carving up the plane*. Let's plot the predicted class at every point in a grid, at several moments during training, and watch the boundary appear out of chaos.

In [ ]:
def plot_boundary(ax, params, X, y, use_relu, title):
    xx, yy = np.meshgrid(np.linspace(-1.5, 1.5, 200), np.linspace(-1.5, 1.5, 200))
    grid = np.c_[xx.ravel(), yy.ravel()]
    logits, _ = forward(params, grid, use_relu=use_relu)
    probs = softmax(logits)[:, 1].reshape(xx.shape)

    cmap = LinearSegmentedColormap.from_list("xor", [PALETTE["rose"], PALETTE["paper"], PALETTE["teal"]])
    ax.contourf(xx, yy, probs, levels=20, cmap=cmap, alpha=0.8)
    ax.contour(xx, yy, probs, levels=[0.5], colors=PALETTE["ink"], linewidths=1.5)
    ax.scatter(X[y==0,0], X[y==0,1], color=PALETTE["rose"], s=18, edgecolor=PALETTE["ink"], linewidth=0.4)
    ax.scatter(X[y==1,0], X[y==1,1], color=PALETTE["teal"], s=18, edgecolor=PALETTE["ink"], linewidth=0.4)
    ax.set_title(title, fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])

fig, axes = plt.subplots(1, len(snap_at), figsize=(4*len(snap_at), 4))
for ax, step in zip(axes, snap_at):
    plot_boundary(ax, snapshots[step], X_train, y_train, use_relu=True, title=f"step {step}")
plt.suptitle("Decision boundary evolving during training (with ReLU)", fontweight="bold")
plt.tight_layout(); plt.show()

You can see the network go from a random guess (step 0) to a slightly tilted line (step 50) to a bent boundary (step 200) to a confident XOR solution (step 1000+). At no point did we tell it "try a curved line" — it just got there by following the gradient.

## The ablation: what if we remove the nonlinearity?

Here's the key demo of the whole section. Let's train the exact same network, but set `use_relu=False`. The network is now a composition of linear layers, which (by matrix multiplication) is itself a single linear layer no matter how many we stack.

In [ ]:
linear_params, linear_hist, linear_snap = train(
    X_train, y_train, steps=2000, lr=0.3, use_relu=False, hidden=16, snapshots_at=[0, 200, 1000]
)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
plot_boundary(axes[0], final_params,  X_train, y_train, use_relu=True,
              title=f"WITH ReLU (loss={loss_hist[-1]:.3f})")
plot_boundary(axes[1], linear_params, X_train, y_train, use_relu=False,
              title=f"NO ReLU — pure linear (loss={linear_hist[-1]:.3f})")
plt.suptitle("With vs without the nonlinearity", fontweight="bold")
plt.tight_layout(); plt.show()

The right-hand plot is damning. Without ReLU, the network draws the best straight line it can through the diagonals and gets ~50% of the points wrong. It physically cannot do better. Stacking more "linear layers" does nothing — it's still a line.

This is why nonlinearities exist. They are *the* thing that lets neural networks represent functions beyond straight lines. Everything else — width, depth, fancy optimizers, GPUs — is in service of this one idea: bend the space.

**Checkpoint:**

1. Change `hidden=16` to `hidden=2`. Can the network still solve XOR? Why?
2. Replace `relu` with a linear function that's just `x` — same result as `use_relu=False`? Why?
3. Derive $\partial L / \partial W_1$ by hand for a batch of size 1. Compare to our matrix form.

---

# 4. Autoencoders — Compress and Reconstruct

Last section. We'll cross the Rubicon from NumPy to PyTorch here, because training a network on real images with manual backprop is a waste of your evening. Everything we built in section 3 is still under the hood — PyTorch just automates the tedious parts.

An autoencoder is a network that tries to learn the identity function the hard way. You give it an input $\mathbf{x}$. It squeezes the input through a tight **bottleneck** — a much smaller vector $\mathbf{z}$ — and then tries to reconstruct the original:

$$
\mathbf{z} = f_{\text{enc}}(\mathbf{x}), \qquad \hat{\mathbf{x}} = f_{\text{dec}}(\mathbf{z}), \qquad L = \|\mathbf{x} - \hat{\mathbf{x}}\|^2
$$

The encoder $f_{\text{enc}}$ compresses. The decoder $f_{\text{dec}}$ reconstructs. The loss is "how different is the reconstruction from the original". Simple.

The interesting part is the bottleneck. If $\mathbf{z}$ is smaller than $\mathbf{x}$, the network is forced to throw away information — but *which* information? It has to keep whatever is most useful for reconstruction, which means it has to learn the structure of the data. An autoencoder trained on digits will learn what "digit-ness" is, because that's the information it can't afford to lose.

This idea — squeeze and reconstruct — is everywhere in ML. It's in VAEs. It's in diffusion models (kind of). It's in JPEG. The attention mechanism in transformers can be understood as a soft bottleneck over the context. Once you see it, you see it everywhere.

## MNIST in one cell

MNIST is the "fruit fly of machine learning." 28x28 grayscale images of handwritten digits, 70k of them. We'll load it from torchvision.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print("using device:", device)

transform = transforms.Compose([transforms.ToTensor()])
train_ds = datasets.MNIST(root="./_mnist", train=True, download=True, transform=transform)
test_ds  = datasets.MNIST(root="./_mnist", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False)

# Peek
x_batch, y_batch = next(iter(train_loader))
print("batch shape:", x_batch.shape)

fig, axes = plt.subplots(1, 8, figsize=(10, 1.8))
for i, ax in enumerate(axes):
    ax.imshow(x_batch[i, 0], cmap="gray")
    ax.set_title(int(y_batch[i]))
    ax.axis("off")
plt.suptitle("sample digits", fontweight="bold")
plt.tight_layout(); plt.show()

## The autoencoder, in ~20 lines

Encoder: 784 → 128 → 32 → `z_dim`. Decoder: `z_dim` → 32 → 128 → 784. We'll pick `z_dim = 2` so we can actually look at the latent space. Squeezing a 784-dimensional image into 2 numbers is aggressive — the reconstructions will be blurry — but that's fine, we care about the geometry of $\mathbf{z}$ here, not pixel perfection.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, z_dim=2):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(28*28, 128), nn.ReLU(),
            nn.Linear(128, 32),    nn.ReLU(),
            nn.Linear(32, z_dim),
        )
        self.dec = nn.Sequential(
            nn.Linear(z_dim, 32),  nn.ReLU(),
            nn.Linear(32, 128),    nn.ReLU(),
            nn.Linear(128, 28*28), nn.Sigmoid(),  # pixels in [0, 1]
        )

    def forward(self, x):
        z = self.enc(x.view(x.size(0), -1))
        xh = self.dec(z).view(x.size(0), 1, 28, 28)
        return xh, z

model = Autoencoder(z_dim=2).to(device)
print(model)
print("params:", sum(p.numel() for p in model.parameters()))

In [ ]:
# Train it — a few epochs is enough to see the idea
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 4

for epoch in range(EPOCHS):
    model.train()
    total = 0.0
    for x, _ in train_loader:
        x = x.to(device)
        xh, _ = model(x)
        loss = F.mse_loss(xh, x)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item() * x.size(0)
    print(f"epoch {epoch+1}/{EPOCHS}  mse={total/len(train_ds):.4f}")

## Reconstructions

Let's grab a batch from the test set and compare originals to reconstructions. Remember, every reconstruction passed through a two-number bottleneck. The network has to recreate 784 pixels from 2 numbers. It can't be good. But it should be recognizable.

In [ ]:
model.eval()
with torch.no_grad():
    x, _ = next(iter(test_loader))
    x = x.to(device)
    xh, z = model(x)

fig, axes = plt.subplots(2, 10, figsize=(12, 2.8))
for i in range(10):
    axes[0, i].imshow(x[i, 0].cpu(),  cmap="gray"); axes[0, i].axis("off")
    axes[1, i].imshow(xh[i, 0].cpu(), cmap="gray"); axes[1, i].axis("off")
axes[0, 0].set_ylabel("original", rotation=0, labelpad=40, va="center")
axes[1, 0].set_ylabel("reconstr.", rotation=0, labelpad=40, va="center")
plt.suptitle("Originals vs reconstructions (z is 2-D)", fontweight="bold")
plt.tight_layout(); plt.show()

Blurry, but the digit identities survive. Not bad for compressing an image into two floats.

## The latent space itself

Here's the payoff. Let's encode the entire test set and scatter each digit in latent space, colored by its true label. If the autoencoder learned anything useful, digits of the same class should cluster.

In [ ]:
all_z, all_y = [], []
model.eval()
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        _, z = model(x)
        all_z.append(z.cpu().numpy())
        all_y.append(y.numpy())
Z = np.concatenate(all_z)
Y = np.concatenate(all_y)

fig, ax = plt.subplots(figsize=(8, 7))
sc = ax.scatter(Z[:,0], Z[:,1], c=Y, cmap="tab10", s=4, alpha=0.7)
cb = plt.colorbar(sc, ticks=range(10))
cb.set_label("digit class")
ax.set_title("MNIST test set, encoded into 2-D latent space", loc="left", fontweight="bold")
ax.set_xlabel("z1"); ax.set_ylabel("z2")
plt.tight_layout(); plt.show()

The clusters are there. They're not perfectly separated — remember we trained the encoder purely to reconstruct, never to classify — but digits of the same class naturally end up near each other, because they look alike and similar-looking things are most efficiently compressed to similar codes. **We never gave the network labels.** The structure fell out for free.

## Walking through latent space

The other beautiful demo: pick any two points in $z$-space and linearly interpolate between them. Decode each intermediate point. Watch one digit morph smoothly into another.

In [ ]:
# Find one example of each digit 0..9 and encode them
examples = {}
model.eval()
with torch.no_grad():
    for x, y in test_loader:
        for xi, yi in zip(x, y):
            yi = int(yi)
            if yi not in examples:
                examples[yi] = xi.to(device)
            if len(examples) == 10:
                break
        if len(examples) == 10:
            break

with torch.no_grad():
    _, z0 = model(examples[3].unsqueeze(0))  # from a 3
    _, z1 = model(examples[8].unsqueeze(0))  # to an 8
    steps = 12
    alphas = np.linspace(0, 1, steps)
    zs = torch.stack([(1-a)*z0 + a*z1 for a in alphas]).squeeze(1)
    recon = model.dec(zs).view(-1, 28, 28).cpu().numpy()

fig, axes = plt.subplots(1, steps, figsize=(12, 1.7))
for i, ax in enumerate(axes):
    ax.imshow(recon[i], cmap="gray")
    ax.axis("off")
plt.suptitle("interpolation: 3  ➜  8", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Bonus: sweep a dense grid of z values and decode each → a visual map of latent space
n = 16
grid = np.linspace(-3, 3, n)
zs = np.array([[a, b] for a in grid for b in grid[::-1]], dtype=np.float32)
with torch.no_grad():
    imgs = model.dec(torch.tensor(zs, device=device)).view(-1, 28, 28).cpu().numpy()

canvas = np.zeros((28*n, 28*n))
for i in range(n):
    for j in range(n):
        canvas[j*28:(j+1)*28, i*28:(i+1)*28] = imgs[i*n + j]

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(canvas, cmap="gray")
ax.set_title("Latent space manifold — every point in (z1, z2) decoded", loc="left", fontweight="bold")
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

Look at that. A continuous map of "digit-space". You can see regions that look like 0s, regions that look like 1s, and transitional areas where digits morph into each other. The network defined a coordinate system over digits — and each digit is a smooth deformation of its neighbors.

This is why people get excited about representation learning. The latent space isn't just a compressed version of the data, it's a *semantic map*. And this exact idea — compressing high-dimensional data into a small, well-behaved latent code — is the backbone of variational autoencoders, diffusion models, and the hidden states of every transformer you've ever talked to.

## A teaser: why VAEs exist

One problem with our autoencoder: the latent space isn't actually *nice*. If you pick a random point $z$ that wasn't in the training set, the decoded image is often garbage. The encoder only mapped training examples to specific points and there's no guarantee the space between them is meaningful — it just happened to be, sometimes, above.

The Variational Autoencoder (VAE) fixes this by forcing the encoder to output a *distribution* over $z$ rather than a single point, and adding a second loss term that pulls that distribution toward a standard normal $\mathcal{N}(0, I)$:

$$
L_{\text{VAE}} = \underbrace{\|\mathbf{x} - \hat{\mathbf{x}}\|^2}_{\text{reconstruct}} \;+\; \underbrace{\beta \cdot D_{\text{KL}}\!\big(q(z|\mathbf{x}) \,\|\, \mathcal{N}(0, I)\big)}_{\text{stay close to Gaussian}}
$$

The KL divergence term measures how far the encoder's output distribution is from a unit Gaussian. Minimizing it forces the latent space to be smooth, continuous, and *sampleable* — you can draw $z \sim \mathcal{N}(0, I)$ and get a valid-looking digit. This is the critical ingredient for *generative* models. A plain autoencoder reconstructs; a VAE generates.

We'll build one properly later in the curriculum. For now just hold the shape of the idea: reconstruction loss + a regularizer that keeps the latent space well-behaved.

**Checkpoint:**

1. Try `z_dim = 8` or `z_dim = 32`. Do reconstructions get sharper? Is the latent space as easy to visualize?
2. Pick two latent points by hand (say `[2.0, 0.5]` and `[-1.0, -2.0]`) and decode them. Do you get recognizable digits?
3. In our plain autoencoder, why do points between two training examples sometimes decode to nonsense?

---

# Wrap-up

That was Part I. Let's take stock.

We started from the dumbest possible place — "computers only understand numbers" — and walked through the whole foundation stack. You implemented BPE, so you now know what happens when you type a message into ChatGPT. You built and used embeddings, so you know what the model sees when it looks up a token. You wrote backprop by hand, so the rest of this curriculum doesn't need to be magic. And you trained an autoencoder, which is a tiny taste of the bottleneck idea that shows up again in transformers, in attention, in diffusion, everywhere.

Part II is where things get spicy: we're going to take the dot product, stack it with a softmax, add a $\sqrt{d_k}$, and invent attention. Which turns out to be the whole game.

See you there.